# Modelado CRISP-DM - Airbnb Trust & Safety

Este notebook documenta el modelado CRISP-DM del proyecto Airbnb usando solamente la informacion disponible en este repositorio: arquitectura medallion, datasets gold, mart de trust/risk, dashboard y documentacion tecnica.

**Objetivo SMART:** desarrollar e implementar, hasta diciembre de 2026, un sistema predictivo de calidad y riesgo de listings basado en machine learning, con precision minima de 85% para identificar listings con alta probabilidad de generar disputas graves antes de su primera reserva, y reducir en al menos 20% la tasa de disputas graves en EE.UU., Brasil, Mexico, Espana y Japon frente a la linea base 2S-2025.

**Pregunta de negocio:** Como reducir la tasa de disputas graves en los 5 paises en un 20%?

## 1. Comprension del negocio

El problema se formula como una iniciativa de Trust & Safety y calidad de listings. La unidad de analisis es `listing_id` y el objetivo operativo es anticipar el riesgo antes de la primera reserva para aplicar acciones preventivas.

Criterios de exito:

- Negocio: reducir en al menos 20% la tasa de disputas graves vs. linea base 2S-2025.
- Modelo: alcanzar precision minima de 85% en listings de alta probabilidad de disputa grave.
- Operacion: priorizar listings `High` y `Critical` para revision, verificacion, mejora de contenido o hold preventivo.
- Medicion: incorporar reservas, disputas y soporte para validar precision, falsos positivos y reduccion real.

## 2. Comprension de los datos

El repositorio usa arquitectura medallion:

```text
raw -> bronze -> silver -> gold
```

Las fuentes actuales principales son:

- `data/gold/listings.parquet`
- `data/gold/reviews.parquet`
- `data/gold/analytics/listing_trust_risk.parquet`
- `data/gold/analytics/market_trust_risk_summary.parquet`
- `data/gold/analytics/risk_segment_summary.parquet`

El proyecto documenta una limitacion clave: la primera review observada se usa como proxy operativo de actividad posterior a reserva. Para medir precision, falsos positivos y reduccion real de disputas graves se requieren tablas de reservas, disputas y soporte.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
paths = {
    "listings_gold": ROOT / "data/gold/listings.parquet",
    "reviews_gold": ROOT / "data/gold/reviews.parquet",
    "listing_trust_risk": ROOT / "data/gold/analytics/listing_trust_risk.parquet",
    "market_summary": ROOT / "data/gold/analytics/market_trust_risk_summary.parquet",
    "segment_summary": ROOT / "data/gold/analytics/risk_segment_summary.parquet",
}

pd.DataFrame(
    [{"asset": name, "path": str(path), "exists": path.exists()} for name, path in paths.items()]
)

In [ ]:
summary_rows = []
for name, path in paths.items():
    if path.exists():
        df = pd.read_parquet(path)
        summary_rows.append({"asset": name, "rows": len(df), "columns": len(df.columns)})
    else:
        summary_rows.append({"asset": name, "rows": None, "columns": None})

pd.DataFrame(summary_rows)

In [ ]:
if paths["market_summary"].exists():
    market_summary = pd.read_parquet(paths["market_summary"])
    display(
        market_summary[
            [
                "country",
                "market_label",
                "listings",
                "avg_risk_score",
                "high_risk_listings",
                "high_risk_rate",
                "critical_risk_listings",
            ]
        ].sort_values("high_risk_rate", ascending=False)
    )

In [ ]:
if paths["segment_summary"].exists():
    segment_summary = pd.read_parquet(paths["segment_summary"])
    display(segment_summary)

## 3. Preparacion de los datos

El pipeline actual ya realiza preparacion basica:

- `bronze`: estandarizacion estructural y metadata de origen.
- `silver`: tipificacion, limpieza, deduplicacion y normalizacion de entidades.
- `gold`: tablas business-ready separadas de listings y reviews.
- `analytics`: mart de riesgo por listing, resumen por mercado y resumen por segmento.

Variables candidatas para modelado:

- Calidad del listing: rating, descripcion, completitud de precio, banos, dormitorios y coordenadas.
- Confianza del host: identidad verificada, superhost, tasa de respuesta, tasa de aceptacion y antiguedad.
- Evidencia de reviews: reviews observadas, longitud promedio de comentario, recencia de reviews.
- Mercado: pais, ciudad/mercado, barrio, coordenadas.
- Oferta: tipo de habitacion, tipo de propiedad, capacidad, noches minimas/maximas, disponibilidad.
- Precio: precio, precio por huesped y posible outlier por mercado.

## 4. Definicion de etiqueta futura

La etiqueta supervisada requerida no existe todavia en el repositorio. Se propone crearla cuando se incorporen tablas de reservas y disputas:

```text
severe_dispute_flag = 1 si el listing genera una disputa grave en la primera reserva o dentro de una ventana operativa definida despues de la primera reserva; 0 en caso contrario.
```

Para cumplir el objetivo de prediccion antes de primera reserva, las features de entrenamiento y scoring deben estar disponibles antes de esa reserva. Cualquier dato posterior solo debe usarse como outcome de evaluacion.

In [ ]:
candidate_feature_groups = {
    "listing_quality": [
        "review_scores_rating",
        "description_length",
        "price",
        "bathrooms",
        "bedrooms",
        "latitude",
        "longitude",
    ],
    "host_trust": [
        "host_identity_verified_bool",
        "host_is_superhost_bool",
        "host_response_rate",
        "host_acceptance_rate",
    ],
    "review_confidence": [
        "observed_reviews",
        "avg_comment_length",
        "review_recency_days",
        "number_of_reviews",
        "reviews_per_month",
    ],
    "market_supply_price": [
        "country",
        "market_label",
        "room_type",
        "property_type",
        "accommodates",
        "minimum_nights",
        "maximum_nights",
        "availability_365",
        "price_per_accommodates",
    ],
}

candidate_feature_groups

## 5. Modelado

Formulacion:

```text
P(disputa_grave = 1 | senales_pre_reserva_del_listing)
```

Baseline actual:

- `interpretable_rule_based_score_v1`
- `risk_score = quality_risk + host_trust_risk + review_confidence_risk + listing_completeness_risk + price_market_risk`
- `trust_score = 100 - risk_score`
- segmentos: `Low` 0-24, `Moderate` 25-49, `High` 50-74, `Critical` 75-100

Escalera de modelos recomendada por la arquitectura del proyecto:

1. Regresion logistica.
2. Random forest.
3. Gradient boosting.
4. Tabular + embeddings de texto.
5. Ensamble.

In [ ]:
if paths["listing_trust_risk"].exists():
    scores = pd.read_parquet(paths["listing_trust_risk"])
    risk_columns = [
        "quality_risk",
        "host_trust_risk",
        "review_confidence_risk",
        "listing_completeness_risk",
        "price_market_risk",
    ]
    display(scores[risk_columns + ["risk_score", "trust_score", "risk_segment"]].head())
    display(scores["risk_segment"].value_counts(dropna=False).rename_axis("risk_segment").reset_index(name="listings"))

## 6. Evaluacion

Metricas tecnicas necesarias:

- `precision`: meta minima 85%.
- `recall`: cobertura de disputas graves detectadas.
- `F1`: balance entre precision y recall.
- `ROC-AUC` y `PR-AUC`: separabilidad del modelo.
- `precision@k` y `recall@k`: utilidad para colas de revision manual.
- `false_positive_rate`: control de impacto injusto sobre hosts.
- calibracion, segment fairness y model drift.

Metrica de negocio:

```text
reduccion_disputas = (tasa_base_2S_2025 - tasa_post_modelo) / tasa_base_2S_2025
```

El objetivo se cumple si `reduccion_disputas >= 20%`.

In [ ]:
def dispute_reduction_rate(baseline_rate: float, post_model_rate: float) -> float:
    """Return relative reduction in severe dispute rate."""
    if baseline_rate <= 0:
        raise ValueError("baseline_rate must be greater than 0")
    return (baseline_rate - post_model_rate) / baseline_rate


# Example: replace with real 2S-2025 baseline and post-model severe dispute rates.
example_baseline_rate = 0.050
example_post_model_rate = 0.040
dispute_reduction_rate(example_baseline_rate, example_post_model_rate)

## 7. Despliegue

Flujo operativo actual:

```bash
python -m src.pipeline.run_medallion
python -m src.pipeline.build_trust_risk_analytics
python -m src.dashboard.trust_risk_dashboard
```

Uso operativo de segmentos:

- `Critical`: hold preventivo o revision inmediata.
- `High`: revision manual y accion correctiva.
- `Moderate`: monitoreo y mejora de informacion.
- `Low`: operacion normal.

Salidas:

- `listing_trust_risk.parquet`: priorizacion por listing.
- `market_trust_risk_summary.parquet`: gestion por mercado.
- `risk_segment_summary.parquet`: seguimiento ejecutivo.
- `trust_risk_dashboard.html`: visualizacion para negocio.

## 8. Plan hasta diciembre 2026

| Periodo | Entregable |
|---|---|
| Meses 1-2 | Integrar contratos de reservas, disputas y soporte; fijar definicion de disputa grave y linea base 2S-2025. |
| Meses 3-4 | Construir snapshots pre-reserva, features historicas y etiqueta `severe_dispute_flag`. |
| Meses 5-6 | Entrenar baseline supervisado y gradient boosting; comparar contra `risk_score` actual. |
| Meses 7-8 | Calibrar umbrales por pais/mercado y validar precision minima de 85%. |
| Meses 9-10 | Piloto operativo en los 5 paises con revision manual y acciones preventivas. |
| Meses 11-12 | Medir reduccion vs 2S-2025, ajustar monitoreo y preparar despliegue productivo. |

## 9. Decision CRISP-DM

Con la informacion actual, el proyecto debe usar el `risk_score` explicable como baseline de priorizacion y preparar el camino hacia un modelo supervisado cuando existan etiquetas reales de disputas graves. No se debe declarar precision de 85% ni reduccion de 20% hasta incorporar reservas, disputas y soporte.

La estrategia para responder la pregunta de negocio es identificar preventivamente listings con mayor riesgo, intervenir antes de la primera reserva y medir el cambio de tasa de disputas graves contra la linea base 2S-2025 en EE.UU., Brasil, Mexico, Espana y Japon.